In [7]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Literal
from langchain_openai.chat_models import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv
load_dotenv()

True

# TABLE OF CONTENTS 

1)  Checkpointer 
    -  getting states and history 
    -  updating states 

# 1) Checkpointer 

In [2]:
class Credit_Score(TypedDict):
    credit_score :  int 
    sanctioned  :  bool
    status :  str 

    
    
model =  ChatOpenAI(model="gpt-4.1-mini")

In [3]:
def show_credit_score(state :  Credit_Score) :
    
    return {"score" : state["credit_score"]}


def check_credit(state :  Credit_Score) -> Literal["excellent_score","fair_score","poor_score"]:
    
    if state["credit_score"]  >  700  :
        return "excellent_score"
    
    elif 650 < state["credit_score"] <  700 :
        return  "fair_score"
    else:
        return "poor_score"
        

def excellent_score(state :  Credit_Score):
    
    
    state["sanctioned"]  =  True
    prompt =  f"Write a message depending on the credit score.  :  {state['credit_score']} and {state['sanctioned']}"
    ans =  model.invoke(prompt)    
    return {"status":ans.content }

def fair_score(state  :  Credit_Score):
    
    
    state["sanctioned"]  =  True
    prompt =  f"Write a message depending on the credit score.  :  {state['credit_score']} and {state['sanctioned']}"
    ans =  model.invoke(prompt)    
    return {"status":ans.content }

def poor_score(state :  Credit_Score):
    
    state["sanctioned"]  =  False
    prompt =  f"Write a message depending on the credit score.  :  {state['credit_score']} and {state['sanctioned']}"
    ans =  model.invoke(prompt)    
    return {"status":ans.content }
    
    



        

In [8]:
graph =  StateGraph(Credit_Score)



graph.add_node("show_credit_score",show_credit_score)
graph.add_node("excellent_score",excellent_score)
graph.add_node("fair_score",fair_score)
graph.add_node("poor_score",poor_score)



# 1) The checking condition will not be addressed as the node 
# 2)  and will return the node name. 

graph.add_edge(START,"show_credit_score")
graph.add_conditional_edges("show_credit_score",check_credit)
graph.add_edge("excellent_score",END)
graph.add_edge("fair_score",END)
graph.add_edge("poor_score",END)


checkpointer =  InMemorySaver()

workflow =  graph.compile(checkpointer=checkpointer)


In [ ]:

config1 =  {"configurable":{"thread_id":1}}
workflow.invoke({"credit_score":800},config=config1)

{'credit_score': 800,
 'status': 'Your credit score is excellent at 800! Keep up the great financial habits!'}

In [ ]:

config2 =  {"configurable":{"thread_id":2}}
workflow.invoke({"credit_score":450},config=config2)

{'credit_score': 450,
 'status': 'Your credit score is 450. Unfortunately, this score is considered poor, which may make it difficult to secure loans or credit at favorable terms. It’s important to review your credit report for any errors, pay down outstanding debts, and make all payments on time to help improve your score over time.'}

## A) Getting states and history 

In [16]:
get_state =  workflow.get_state(config1)
print(get_state)

get_state_history =  workflow.get_state_history(config=config1)
print(list(get_state_history))

StateSnapshot(values={'credit_score': 800, 'status': 'Your credit score is excellent at 800! Keep up the great financial habits!'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d60dc-bd25-6d4b-8002-e1449f47ddbe'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-11-01T16:03:36.767623+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d60dc-ad8c-63c1-8001-fe1e836b27de'}}, tasks=(), interrupts=())
[StateSnapshot(values={'credit_score': 800, 'status': 'Your credit score is excellent at 800! Keep up the great financial habits!'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d60dc-bd25-6d4b-8002-e1449f47ddbe'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-11-01T16:03:36.767623+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d60dc-ad8c-63c1-8001

# B) Updating state

In [19]:
workflow.update_state({"configurable":{"thread_id":1,"checkpoint_id":"1f1d60dc-bd25-6d4b-8002-e1449f47ddbe","checkpoint_ns":""}},{"credit_score":230})

get_state =  workflow.get_state(config1)
print(get_state)

StateSnapshot(values={'credit_score': 230, 'status': 'Your credit score is excellent at 800! Keep up the great financial habits!'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d60e9-85c0-69e5-8003-574b503e15db'}}, metadata={'source': 'update', 'step': 3, 'parents': {}}, created_at='2026-11-01T16:09:19.925091+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d60dc-bd25-6d4b-8002-e1449f47ddbe'}}, tasks=(), interrupts=())
